In [4]:

import pandas as pd
import win32com.client as win32
import pythoncom
from pathlib import Path
from datetime import datetime
from html import escape
import re
import time

# =========================================================
# 1. SETTINGS
# =========================================================

folder_path = Path(r"C:\Users\hp\OneDrive\Desktop\DEFERRALS")

report_folder = folder_path / "Branch_Deferral_Reports"
report_folder.mkdir(parents=True, exist_ok=True)

REGIONAL_MANAGER_EMAIL = "mutugikenneth28@gmail.com"

# False = Create Outlook drafts for review
# True = Send emails
SEND_EMAILS = True

# =========================================================
# 2. ACTUAL EXCEL COLUMN NAMES
# =========================================================

BRANCH = "BRANCH"
RM_NAME = "REQUESTER/RM"
RM_EMAIL = "RM Email"
BM_EMAIL = "BM Email"

BORROWER = "Borrower"
CUSTOMER_NUMBER = "Customer Number"
DEFERRED_ITEM = "Item Deferred"
DUE_DATE = "Due Date"
STATUS = "Status"
AGEING = "Ageing (Days)"
OVERDUE_STATUS = "Overdue Statu"

required_columns = [
    BRANCH,
    RM_NAME,
    RM_EMAIL,
    BM_EMAIL,
    BORROWER,
    CUSTOMER_NUMBER,
    DEFERRED_ITEM,
    DUE_DATE,
    STATUS
]

# =========================================================
# 3. FETCH THE LATEST EXCEL FILE
# =========================================================

excel_files = [
    f for f in folder_path.glob("*.xlsx")
    if not f.name.startswith("~$")
]

if not excel_files:
    raise FileNotFoundError(
        "No Excel files found in the DEFERRALS folder."
    )

latest_file = max(
    excel_files,
    key=lambda f: f.stat().st_mtime
)

print("Latest Excel file:", latest_file.name)

df = pd.read_excel(latest_file)

# Remove accidental spaces in column headings
df.columns = df.columns.astype(str).str.strip()

# =========================================================
# 4. CHECK REQUIRED COLUMNS
# =========================================================

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"These columns are missing from your Excel file: "
        f"{missing_columns}\n\n"
        f"Available columns: {df.columns.tolist()}"
    )

# =========================================================
# 5. CLEAN EMAIL ADDRESSES
# =========================================================

def clean_email(value):

    if pd.isna(value):
        return ""

    email = str(value).strip()

    # Handles Excel values copied in Markdown link format
    if "mailto:" in email.lower():
        match = re.search(
            r"mailto:([^\])\s]+)",
            email,
            flags=re.IGNORECASE
        )
        if match:
            email = match.group(1)

    email = email.replace("[", "").replace("]", "")
    email = email.split("(")[0].strip()

    if email.lower() in ["", "nan", "none"]:
        return ""

    return email


df[RM_EMAIL] = df[RM_EMAIL].apply(clean_email)
df[BM_EMAIL] = df[BM_EMAIL].apply(clean_email)

# =========================================================
# 6. FILTER PENDING DEFERRALS ONLY
# =========================================================

df[STATUS] = df[STATUS].fillna("").astype(str).str.strip()

pending_df = df[
    df[STATUS].str.lower() == "pending"
].copy()

pending_df = pending_df.dropna(subset=[BRANCH])

if pending_df.empty:
    print("There are no pending deferrals.")
    raise SystemExit

# =========================================================
# 7. CALCULATE DAYS OVERDUE
# =========================================================

today = pd.Timestamp.today().normalize()

# Your Excel dates are in DD/MM/YYYY format
pending_df[DUE_DATE] = pd.to_datetime(
    pending_df[DUE_DATE],
    errors="coerce",
    dayfirst=True
)

pending_df["Days Overdue"] = (
    today - pending_df[DUE_DATE].dt.normalize()
).dt.days

# Items not yet due show zero
pending_df["Days Overdue"] = (
    pending_df["Days Overdue"]
    .clip(lower=0)
    .fillna(0)
    .astype(int)
)

# Identify whether each item is overdue
pending_df["Overdue Status"] = pending_df[
    "Days Overdue"
].apply(
    lambda x: "Overdue" if x > 0 else "Not Yet Due"
)

# =========================================================
# 8. FUNCTION TO GET RM FIRST NAME
# =========================================================

def get_first_name(full_name):

    if pd.isna(full_name):
        return "Team"

    name = str(full_name).strip()

    if not name:
        return "Team"

    # Handles names such as Njeru, Kenneth
    if "," in name:
        parts = [x.strip() for x in name.split(",")]
        name = parts[1] if len(parts) > 1 else parts[0]

    return name.split()[0].title()


# =========================================================
# 9. FUNCTION TO CREATE THE EMAIL SUMMARY TABLE
# =========================================================

def create_email_table(branch_df):

    summary = branch_df[
        [
            BORROWER,
            CUSTOMER_NUMBER,
            DEFERRED_ITEM,
            DUE_DATE,
            STATUS,
            "Days Overdue"
        ]
    ].copy()

    summary = summary.rename(columns={
        BORROWER: "Borrower",
        CUSTOMER_NUMBER: "Customer Number",
        DEFERRED_ITEM: "Deferred Item",
        DUE_DATE: "Due Date",
        STATUS: "Status"
    })

    summary["Due Date"] = pd.to_datetime(
        summary["Due Date"],
        errors="coerce"
    ).dt.strftime("%d-%b-%Y")

    summary["Due Date"] = summary["Due Date"].fillna("Not specified")

    html_table = summary.to_html(
        index=False,
        border=0,
        escape=True
    )

    html_table = html_table.replace(
        "<table ",
        '<table style="border-collapse:collapse;'
        'width:100%;font-family:Arial;font-size:10pt;" '
    )

    html_table = html_table.replace(
        "<th>",
        '<th style="background-color:#D9EAF7;'
        'border:1px solid #B7C9D6;padding:7px;text-align:left;">'
    )

    html_table = html_table.replace(
        "<td>",
        '<td style="border:1px solid #B7C9D6;padding:7px;">'
    )

    return html_table


# =========================================================
# 10. CONNECT TO OUTLOOK
# =========================================================

pythoncom.CoInitialize()

outlook = win32.Dispatch("Outlook.Application")

email_log = []

report_date = datetime.now().strftime("%d %B %Y")

# =========================================================
# 11. PROCESS EACH BRANCH
# =========================================================

for branch, branch_df in pending_df.groupby(BRANCH):

    branch_name = str(branch).strip()

    print("\nProcessing branch:", branch_name)

    # Get unique BM email addresses for this branch
    bm_emails = sorted({
        email
        for email in branch_df[BM_EMAIL].tolist()
        if email
    })

    # =====================================================
    # 12. CREATE ONE EXCEL REPORT FOR THE ENTIRE BRANCH
    # =====================================================

    safe_branch = re.sub(
        r"[^A-Za-z0-9_-]",
        "_",
        branch_name
    )

    attachment_path = (
        report_folder /
        f"Outstanding_Deferrals_{safe_branch}_{datetime.now():%Y%m%d}.xlsx"
    )

    with pd.ExcelWriter(
        attachment_path,
        engine="openpyxl"
    ) as writer:

        branch_df.to_excel(
            writer,
            sheet_name="Pending Deferrals",
            index=False
        )

        ws = writer.sheets["Pending Deferrals"]

        ws.freeze_panes = "A2"
        ws.auto_filter.ref = ws.dimensions

        for column_cells in ws.columns:

            max_length = max(
                len(str(cell.value))
                if cell.value is not None else 0
                for cell in column_cells
            )

            ws.column_dimensions[
                column_cells[0].column_letter
            ].width = min(max_length + 2, 40)

    # =====================================================
    # 13. PREPARE BRANCH SUMMARY
    # =====================================================

    email_table = create_email_table(branch_df)

    total_deferrals = len(branch_df)

    overdue_count = (
        branch_df["Days Overdue"] > 0
    ).sum()

    # =====================================================
    # 14. IDENTIFY EACH RM IN THE BRANCH
    # =====================================================

    rm_list = branch_df[
        [RM_NAME, RM_EMAIL]
    ].dropna().drop_duplicates()

    for _, rm in rm_list.iterrows():

        rm_name = str(rm[RM_NAME]).strip()
        rm_email = clean_email(rm[RM_EMAIL])

        if not rm_email:
            print("Skipping RM without email:", rm_name)
            continue

        first_name = get_first_name(rm_name)

        # =================================================
        # 15. CREATE INDIVIDUAL EMAIL
        # =================================================

        mail = outlook.CreateItem(0)

        # Address directly to the RM
        mail.To = rm_email

        # Copy BM and Regional Manager
        cc_emails = bm_emails + [
            REGIONAL_MANAGER_EMAIL
        ]

        # Remove duplicate addresses
        cc_emails = list(dict.fromkeys(cc_emails))

        mail.CC = ";".join(cc_emails)

        mail.Subject = (
            f"Action Required: Outstanding Deferrals - "
            f"{branch_name} - {report_date}"
        )

        mail.HTMLBody = f"""
        <html>
        <body style="font-family:Arial;font-size:11pt;">

        <p>Dear {escape(first_name)},</p>

        <p>
        Please find attached the consolidated outstanding
        deferrals report for <b>{escape(branch_name)}</b>.
        </p>

        <p>
        The branch currently has <b>{total_deferrals}</b>
        pending deferrals, of which <b>{overdue_count}</b>
        are overdue.
        </p>

        <p><b>Outstanding Deferrals Summary</b></p>

        {email_table}

        <p>
        Kindly review the attached Excel report and follow up
        on all outstanding items under your portfolio.
        </p>

        <p><b>Please prioritise the following:</b></p>

        <ul>
            <li>Deferrals that are overdue or approaching their due dates.</li>
            <li>Following up with customers to obtain outstanding documents.</li>
            <li>Submitting the required documents for closure.</li>
            <li>Providing progress updates and expected resolution dates.</li>
        </ul>

        <p>
        Kindly work closely with your Branch Manager to ensure
        timely follow-up and resolution of all outstanding items.
        </p>

        <p>
        Please share your progress updates and expected
        closure dates.
        </p>

        <p>
        Regards,<br>
        Credit Monitoring Team
        </p>

        </body>
        </html>
        """

        # Attach the complete branch Excel report
        mail.Attachments.Add(str(attachment_path))

        # =================================================
        # 16. SAVE DRAFT OR SEND EMAIL
        # =================================================

        try:

            if SEND_EMAILS:
                mail.Send()
                email_status = "Sent"
            else:
                mail.Save()
                email_status = "Draft created"

            print(
                f"{email_status}: {first_name} "
                f"| {branch_name} | {rm_email}"
            )

            email_log.append({
                "Branch": branch_name,
                "RM Name": rm_name,
                "RM Email": rm_email,
                "BM Email": ";".join(bm_emails),
                "Regional Manager": REGIONAL_MANAGER_EMAIL,
                "Total Pending Deferrals": total_deferrals,
                "Overdue Deferrals": overdue_count,
                "Attachment": str(attachment_path),
                "Email Status": email_status
            })

        except Exception as e:

            print("Email error:", e)

            email_log.append({
                "Branch": branch_name,
                "RM Name": rm_name,
                "RM Email": rm_email,
                "Email Status": f"Failed: {e}"
            })

        time.sleep(2)

# =========================================================
# 17. SAVE EMAIL LOG
# =========================================================

log_df = pd.DataFrame(email_log)

log_path = (
    report_folder /
    f"Deferral_Email_Log_{datetime.now():%Y%m%d_%H%M}.xlsx"
)

log_df.to_excel(log_path, index=False)

print("\n======================================")
print("DEFERRAL EMAIL PROCESS COMPLETED")
print("======================================")

print("Register used:", latest_file.name)
print("Emails processed:", len(email_log))
print("Email log:", log_path)
print("Branch reports:", report_folder)


Latest Excel file: Deferral_LIST AS AT 11.19.2026.xlsx

Processing branch: EMBAKASI BRANCH
Sent: Kenneth | EMBAKASI BRANCH | kenneth.njeru@ncbagroup.com

Processing branch: KAREN BRANCH
Sent: Kelvin | KAREN BRANCH | kelvin.yego@ncbagroup.com

Processing branch: RUAKA BRANCH
Sent: Kenneth | RUAKA BRANCH | kenneth.njeru@ncbagroup.com

Processing branch: RUIRU BRANCH
Sent: Shawn | RUIRU BRANCH | shawn.irungu@ncbagroup.com

Processing branch: SWISSPORT BRANCH, NAIROBI
Sent: Kenneth | SWISSPORT BRANCH, NAIROBI | kenneth.njeru@ncbagroup.com

Processing branch: THIKA ROAD MALL BRANCH
Sent: Kenneth | THIKA ROAD MALL BRANCH | kenneth.njeru@ncbagroup.com
Sent: Kelvin | THIKA ROAD MALL BRANCH | kelvin.yego@ncbagroup.com

Processing branch: TWO RIVERS BRANCH
Sent: Shawn | TWO RIVERS BRANCH | shawn.irungu@ncbagroup.com

Processing branch: VILLAGE MARKET BRANCH
Sent: Kenneth | VILLAGE MARKET BRANCH | kenneth.njeru@ncbagroup.com

Processing branch: WESTLANDS BRANCH
Sent: Kenneth | WESTLANDS BRANCH | 